In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


##Summarization Middleware

In [5]:
import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Load environment variables
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize the model
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key,
)

# Create the agent
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)

In [6]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [7]:
# Alternative test data
questions= [
    "What is 2+2?",
    "what is 10*5?",
    "what is 100%4?",
    # "what is 3*3?"
]
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response['messages'])}")


Messages:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='97b99631-5973-4eb9-8742-90b0d822c084'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.006954695, 'completion_tokens_details': None, 'prompt_time': 0.008454453, 'prompt_tokens_details': None, 'queue_time': 0.008395461, 'total_time': 0.015409148}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7a0a-dcdd-7d50-9d56-a08b9c7b97b5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages:2
Messages:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='97b99631-5973-4eb9-8742-90b0d822c084'), AIMessage

Token Size

In [1]:
import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage


# Load environment variables
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize Groq model
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key,
)

# Tool
@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi
    """

# Create agent
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Approximate token counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # Approximation: 4 characters ≈ 1 token

In [2]:
# Run test
cities = ["Paris", "London"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~248 tokens, 10 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='2ca1b6bb-b19b-4bee-9d30-807858d1c042'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'y9x6asd4j', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 226, 'total_tokens': 241, 'completion_time': 0.039216448, 'completion_tokens_details': None, 'prompt_time': 0.012282108, 'prompt_tokens_details': None, 'queue_time': 0.051644052, 'total_time': 0.051498556}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7f96-1e6a-75a0-9b2a-d6e27a81872c-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'y9x6asd4j', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metad

Fraction


In [2]:
middleware=[
    SummarizationMiddleware(
        model=model,
        trigger=("fraction", 0.01),
        keep=("fraction", 0.005),
    ),
],